In [1]:
import pandas as pd
import numpy as np
from plotly import express as pep
iris=pep.data.iris()

合并数据 concat

pd.concat(objs, axis=0, join='outer',
          ignore_index=False, keys=None,
          levels=None, names=None, sort=False,
          verify_integrity=False, copy=True)

- objs 连接数据
- axis 0行 1列
- join 合并方式 inner保留相同列/outer保留两个表都有的列
- keys 连接关系,给原来的数据打标签生成多层索引 (区分数据来源)
- name 索引名称
- verity_integrity 检查是够有重复内容,如果合并数据与原数据包含索引相同行报错


> 一次行合并,减少开销;axis=1先重制索引,如果拼不上会乱;修改列名一样

In [26]:
df1=pd.DataFrame({'A':np.random.randint(1,10,(10,)),'AA':'A'})
df2=pd.DataFrame({'A':np.random.randint(10,20,(10,)),'AA':'B'})
df3=pd.DataFrame({'C':np.random.randint(10,20,(10,)),'AA':'D'},index=pd.date_range('2021-01-01',freq='D',periods=10))

In [ ]:
pd.concat([df1,df2],verify_integrity=True) # 索引相同行

In [ ]:
print(pd.concat([df1,df2]))

In [27]:
pd.concat([df1,df3],keys=['a','b']).sample(5)

A AA     C
b 2021-01-03 00:00:00  NaN  D  16.0
  2021-01-09 00:00:00  NaN  D  18.0
a 0                    8.0  A   NaN
b 2021-01-06 00:00:00  NaN  D  13.0
  2021-01-01 00:00:00  NaN  D  13.0

In [28]:
pd.concat([df1,df3],join='inner').sample(5)

,AA
2021-01-07 00:00:00,D
5,A
2021-01-05 00:00:00,D
2021-01-09 00:00:00,D
4,A


数据合并
类似于SQL中的join操作

pd.merge(left, right, how='inner', on=None, left_on=None, right_on=None,
         left_index=False, right_index=False, sort=True,
         suffixes=('_x', '_y'), copy=True, indicator=False,
         validate=None)

- left/right 需要连接的df,s
- how 怎么连接inner,outer,left,right
    - inner保留左右表共同拥有
    - left/right 左/右表中所有内容保留
    - outer 左右都保留,关联不上的内容Nan
- on 作为连接的字段,左右表都必须存在,否则使用left_on,right_on;如果每指定会自动找相同的连接;指定多键连接
- left_on/right_on 左右表连接字段
- left_index/right_index 将左表索引作为连接键
- indicator 连接提示.连接后的数据左还是右或者都有
- suffixes 如果出现重复列,新数据表会用此后缀区分


In [32]:
df3 = pd.DataFrame({'a': [1, 2], 'b': [3, 4], 'x': [5, 6]})
df4 = pd.DataFrame({'a': [1, 2, 3], 'b': [3, 1, 5],  'y': [6, 7, 8]})
pd.merge(df3, df4, on=['a','b'],how='right',indicator=True)


,a,b,x,y,_merge
0,1,3,5.0,6,both
1,2,1,NaN,7,right_only
2,3,5,NaN,8,right_only


In [33]:
pd.merge(df3, df4, right_index=True,left_index=True)

,a_x,b_x,x,a_y,b_y,y
0,1,3,5,1,3,6
1,2,4,6,2,1,7


In [34]:
pd.merge(df3, df4, on=['a','b'])

,a,b,x,y
0,1,3,5,6


元素合并
combine/combine_first 它们不是像concat堆叠,而是把两个df对齐到同一个坐标系,相同的行列索引,针对每个格子进行合并(ps里的图像叠加)

combine_first用B表补A表中的Nan,A表中的值不动
1. 使用pd.Series({'a':3})利用传播极致补齐数据
2. combine_first 主数据+增量更新

combine不满足补空,还可以传入自定义函数,让两表进行对比然后决定选那个
- func 接受(数据表列)作为参数,返回一列作为结果
- fill_value Nan填充
- overwrite 决定处理只在原表中存在,而不在目标表中存在的列,True以目标表为准
1. func=lambda s1,s2:s1+s2,fill_values=0 解决Nan的问题
2. df_a.combine(df_b,func=np.minimum,overwrite=False)
3. df_a.combine(df_b, func=lambda row_a, row_b: row_a if row_a.sum() > row_b.sum() else row_b, axis=1) 行聚合
4. combine是聚合运算,只用来检查+-比较慢

update 强行覆盖
- join 对齐方式left保留原表索引,inner只取交集
- overwrite 用other里非Nan覆盖,false只覆盖原表本身nan的位置
- filter_func 对满足条件更新,func(1d)A表的列
- errors ignore/raise 当索引列对不齐是保持
> 没有返回值

In [3]:
# A 表：有残缺（缺了 2023-01-01 的 B 列，和 2023-01-02 的 A 列）
df_a = pd.DataFrame({
    'A': [1, 1, 3],
    'B': [np.nan, 5, 6]
}, index=['2023-01-01', '2023-01-02', '2023-01-03'])

# B 表：提供了补丁数据
df_b = pd.DataFrame({
    'A': [10, 20, 30],
    'B': [2, np.nan, 10]
}, index=['2023-01-01', '2023-01-02', '2023-01-03'])

# 用 B 补 A
result = df_a.combine_first(df_b)
result

,A,B
2023-01-01,1,2.0
2023-01-02,1,5.0
2023-01-03,3,6.0


In [4]:
# 
df_a.combine(df_b,func= np.fmax)

,A,B
2023-01-01,10,2.0
2023-01-02,20,5.0
2023-01-03,30,10.0


In [16]:
df_a = pd.DataFrame({
    'A': [1, 1, 3],
    'B': [np.nan, 5, 6],
    'C': 3
}, index=['2023-01-01', '2023-01-02', '2023-01-03'])

df_b = pd.DataFrame({
    'A': [10, 20, 30],
    'B': [14, np.nan, 10]
}, index=['2023-01-01', '2023-01-02', '2023-01-03'])
# fillna触发自动索引对齐
df_a.combine(df_b, func=lambda x1,x2:x2.fillna(x1), overwrite=True)
# df_a.update(df_b,overwrite=False)

# df_a

,A,B,C
2023-01-01,10,14.0,3.0
2023-01-02,20,5.0,3.0
2023-01-03,30,10.0,3.0


In [17]:
def func(a,b):
    print(type(a),a)
    return a
df_a.combine(df_b,func=func,overwrite=False)

<class 'pandas.core.series.Series'> 2023-01-01    1
2023-01-02    1
2023-01-03    3
Name: A, dtype: int64
<class 'pandas.core.series.Series'> 2023-01-01    NaN
2023-01-02    5.0
2023-01-03    6.0
Name: B, dtype: float64


,A,B,C
2023-01-01,1,NaN,3
2023-01-02,1,5.0,3
2023-01-03,3,6.0,3


In [ ]:
df_a = pd.DataFrame({
    'A': [1, np.nan, 3],
    'B': [np.nan, 5, 6]
}, index=['2023-01-01', '2023-01-02', '2023-01-03'])

# B 表：提供了补丁数据
df_b = pd.DataFrame({
    'A': [10, 20, 30],
    'B': [10, np.nan, 11]
}, index=['2023-01-01', '2023-01-02', '2023-01-03'])


def f(x):
    print(type(x),x)
    return False
df_a.update(df_b,filter_func=f)
df_a

<class 'numpy.ndarray'> [ 1. nan  3.]
<class 'numpy.ndarray'> [nan  5.  6.]


,A,B
2023-01-01,1.0,NaN
2023-01-02,NaN,5.0
2023-01-03,3.0,6.0


比较差异
compare
- align_axis
- keep_equal 对比时将相同值显示
- keep_shape 保持形状,False会输出不相同的

> 1. + stack堆积成列
> 2 清洗数据与原数据对比

In [104]:
df1 = pd.DataFrame({'A': [1, 2, 3], 'B': [4, 5, 6]})
df2 = pd.DataFrame({'A': [1, 20, 3], 'B': [40, 5, 6]})

# 默认对齐 axis=1（差异展示在列上）
result = df1.compare(df2,keep_equal=True,keep_shape=True).stack()
print(result)

          A   B
0 self    1   4
  other   1  40
1 self    2   5
  other  20   5
2 self    3   6
  other   3   6


/var/folders/f7/rx7cmvdx6nl86z_s7s86w4zm0000gn/T/ipykernel_1063/538531865.py:5: FutureWarning:

The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.

